In [ ]:
# End-to-End E-commerce Intelligence System
## Building a Customer 360 Analytics Framework

**Objective:** Integrate multiple e-commerce data sources into a unified analytical dataset, construct a Customer 360 view, analyze customer behavior, revenue patterns, and operational performance, and generate actionable business recommendations.

---
## Step 1: Data Loading and Initial Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = './'

### 1.1 Load All Datasets

In [ ]:
customers = pd.read_csv(f'{DATA_DIR}customers.csv')
orders = pd.read_csv(f'{DATA_DIR}orders.csv')
order_items = pd.read_csv(f'{DATA_DIR}order_item.csv')
payments = pd.read_csv(f'{DATA_DIR}payments.csv')
reviews = pd.read_csv(f'{DATA_DIR}reviews.csv')
products = pd.read_csv(f'{DATA_DIR}products.csv')
sellers = pd.read_csv(f'{DATA_DIR}sellers.csv')
geolocation = pd.read_csv(f'{DATA_DIR}location.csv')
category_translation = pd.read_csv(f'{DATA_DIR}category_translation.csv')

datasets = {
    'customers': customers,
    'orders': orders,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation': geolocation,
    'category_translation': category_translation
}

print("Datasets loaded successfully!\n")
for name, df in datasets.items():
    print(f"{name:>25s}: {df.shape[0]:>10,} rows x {df.shape[1]:>3} columns")

### 1.2 Inspect Each Dataset — Structure, Schema & Sample Rows

In [ ]:
for name, df in datasets.items():
    print("=" * 80)
    print(f"  Dataset: {name.upper()}")
    print("=" * 80)
    print(f"\nShape: {df.shape}")
    print(f"\nColumn dtypes:\n{df.dtypes}\n")
    print(f"Missing values:\n{df.isnull().sum()}\n")
    display(df.head(3))
    print("\n")

### 1.3 Statistical Summary of Numerical Columns

In [ ]:
for name, df in datasets.items():
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if numeric_cols:
        print(f"\n--- {name.upper()} ---")
        display(df[numeric_cols].describe())

### 1.4 Identify Primary and Foreign Keys & Table Relationships

| Relationship | Left Table | Key | Right Table |
|---|---|---|---|
| Orders ↔ Customers | orders | `customer_id` | customers |
| Orders ↔ Order Items | orders | `order_id` | order_items |
| Order Items ↔ Products | order_items | `product_id` | products |
| Order Items ↔ Sellers | order_items | `seller_id` | sellers |
| Orders ↔ Payments | orders | `order_id` | payments |
| Orders ↔ Reviews | orders | `order_id` | reviews |
| Products ↔ Category Translation | products | `product_category_name` | category_translation |

The **orders** table serves as the central fact table connecting most datasets.

In [ ]:
print("Key uniqueness check:")
print(f"  customers.customer_id unique: {customers['customer_id'].nunique()} / {len(customers)}")
print(f"  orders.order_id unique:       {orders['order_id'].nunique()} / {len(orders)}")
print(f"  products.product_id unique:   {products['product_id'].nunique()} / {len(products)}")
print(f"  sellers.seller_id unique:     {sellers['seller_id'].nunique()} / {len(sellers)}")
print(f"\n  order_items per order: {order_items.groupby('order_id').size().describe().to_string()}")
print(f"\n  payments per order:    {payments.groupby('order_id').size().describe().to_string()}")

---
## Step 2: Data Cleaning and Preprocessing

### 2.1 Handle Missing Values

In [ ]:
print("Missing values across all datasets:\n")
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"  {name}:")
        for col, cnt in missing.items():
            pct = cnt / len(df) * 100
            print(f"    {col:>40s}: {cnt:>6,} ({pct:.2f}%)")
        print()
    else:
        print(f"  {name}: No missing values\n")

In [ ]:
# Orders: delivery dates are NaN for orders not yet delivered — expected behavior, keep as-is
# Reviews: comment title & message are optional; fill with empty strings
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('')

# Products: fill missing category with 'unknown', numeric dimensions with median
products['product_category_name'] = products['product_category_name'].fillna('unknown')
product_numeric_cols = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty',
                        'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in product_numeric_cols:
    products[col] = products[col].fillna(products[col].median())

print("Missing values handled successfully.")

### 2.2 Remove Duplicate Records

In [ ]:
print("Duplicate rows in each dataset:")
for name, df in datasets.items():
    dup_count = df.duplicated().sum()
    print(f"  {name:>25s}: {dup_count:>6,} duplicates")

# Geolocation has many duplicates (same zip with slightly different coordinates); deduplicate
geolocation = geolocation.drop_duplicates(subset=['geolocation_zip_code_prefix'], keep='first')
datasets['geolocation'] = geolocation
print(f"\nGeolocation after dedup: {len(geolocation):,} rows")

### 2.3 Convert Date Columns to Datetime Format

In [ ]:
order_date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in order_date_cols:
    orders[col] = pd.to_datetime(orders[col])

order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

print("Date columns converted:")
print(orders[order_date_cols].dtypes)

### 2.4 Validate Data Types and Ranges

In [ ]:
print("Order statuses:", orders['order_status'].value_counts().to_dict())
print(f"\nPayment types: {payments['payment_type'].unique()}")
print(f"Review scores range: {reviews['review_score'].min()} - {reviews['review_score'].max()}")
print(f"Price range: R${order_items['price'].min():.2f} - R${order_items['price'].max():.2f}")
print(f"Freight range: R${order_items['freight_value'].min():.2f} - R${order_items['freight_value'].max():.2f}")
print(f"Payment value range: R${payments['payment_value'].min():.2f} - R${payments['payment_value'].max():.2f}")
print(f"Order date range: {orders['order_purchase_timestamp'].min()} to {orders['order_purchase_timestamp'].max()}")

---
## Step 3: Data Integration — Constructing the Master Dataset

Merge sequence:
1. `orders` + `customers` (on `customer_id`)
2. Result + `order_items` (on `order_id`)
3. Result + `products` (on `product_id`)
4. Result + `category_translation` (on `product_category_name`)
5. Result + `sellers` (on `seller_id`)
6. Result + `payments` (on `order_id`)
7. Result + `reviews` (on `order_id`)

In [ ]:
# 1. Orders + Customers
master = orders.merge(customers, on='customer_id', how='left')
print(f"After orders + customers: {master.shape}")

# 2. + Order Items
master = master.merge(order_items, on='order_id', how='left')
print(f"After + order_items:      {master.shape}")

# 3. + Products
master = master.merge(products, on='product_id', how='left')
print(f"After + products:         {master.shape}")

# 4. + Category Translation
master = master.merge(category_translation, on='product_category_name', how='left')
print(f"After + category_transl.: {master.shape}")

# 5. + Sellers
master = master.merge(sellers, on='seller_id', how='left')
print(f"After + sellers:          {master.shape}")

# 6. + Payments (aggregated per order to avoid row explosion)
payments_agg = payments.groupby('order_id').agg(
    payment_value_total=('payment_value', 'sum'),
    payment_installments_max=('payment_installments', 'max'),
    payment_type_primary=('payment_type', 'first'),
    payment_count=('payment_type', 'count')
).reset_index()
master = master.merge(payments_agg, on='order_id', how='left')
print(f"After + payments (agg):   {master.shape}")

# 7. + Reviews (take the first review per order)
reviews_dedup = reviews.sort_values('review_creation_date').drop_duplicates(subset='order_id', keep='last')
master = master.merge(
    reviews_dedup[['order_id', 'review_score', 'review_comment_title', 'review_comment_message',
                   'review_creation_date', 'review_answer_timestamp']],
    on='order_id', how='left'
)
print(f"After + reviews:          {master.shape}")

print(f"\nFinal Master Dataset: {master.shape[0]:,} rows x {master.shape[1]} columns")

In [ ]:
print("Master Dataset Columns:\n")
print(master.dtypes.to_string())
print(f"\nMissing values summary:")
missing = master.isnull().sum()
print(missing[missing > 0].to_string())
display(master.head())

---
## Step 4: Feature Engineering

In [ ]:
# --- Order-level features ---

# Total order value (price + freight per item row)
master['order_item_total'] = master['price'] + master['freight_value']

# Delivery time in days (purchase to actual delivery)
master['delivery_time_days'] = (
    master['order_delivered_customer_date'] - master['order_purchase_timestamp']
).dt.total_seconds() / 86400

# Estimated vs actual delivery difference (positive = delivered early)
master['delivery_delay_days'] = (
    master['order_estimated_delivery_date'] - master['order_delivered_customer_date']
).dt.total_seconds() / 86400

# Number of items per order
items_per_order = order_items.groupby('order_id')['order_item_id'].count().rename('items_per_order')
master = master.merge(items_per_order, on='order_id', how='left')

# Order month and year for time-series analysis
master['order_month'] = master['order_purchase_timestamp'].dt.to_period('M')
master['order_year'] = master['order_purchase_timestamp'].dt.year
master['order_day_of_week'] = master['order_purchase_timestamp'].dt.day_name()
master['order_hour'] = master['order_purchase_timestamp'].dt.hour

print("Order-level features created.")

In [ ]:
# --- Customer-level features ---

customer_features = master.groupby('customer_unique_id').agg(
    purchase_count=('order_id', 'nunique'),
    total_spend=('payment_value_total', 'sum'),
    avg_order_value=('payment_value_total', 'mean'),
    avg_review_score=('review_score', 'mean'),
    avg_delivery_time=('delivery_time_days', 'mean'),
    first_purchase=('order_purchase_timestamp', 'min'),
    last_purchase=('order_purchase_timestamp', 'max'),
    total_items_bought=('order_item_id', 'count'),
    unique_categories=('product_category_name_english', 'nunique'),
    customer_state=('customer_state', 'first'),
    customer_city=('customer_city', 'first')
).reset_index()

# Customer lifetime value approximation (total spend)
customer_features['clv'] = customer_features['total_spend']

# Customer tenure in days
customer_features['tenure_days'] = (
    customer_features['last_purchase'] - customer_features['first_purchase']
).dt.total_seconds() / 86400

# Repeat customer flag
customer_features['is_repeat_customer'] = (customer_features['purchase_count'] > 1).astype(int)

print(f"Customer features table: {customer_features.shape}")
display(customer_features.head())

In [ ]:
print("Feature summary statistics:")
display(customer_features[['purchase_count', 'total_spend', 'avg_order_value',
                            'avg_review_score', 'avg_delivery_time', 'clv',
                            'tenure_days', 'total_items_bought']].describe())

---
## Step 5 & 6: Exploratory Data Analysis (EDA) and Visualization

### 5.1 Customer Analysis

#### 5.1.1 New vs Repeat Customers

In [ ]:
repeat_counts = customer_features['is_repeat_customer'].value_counts()
repeat_labels = ['One-time Customer', 'Repeat Customer']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(repeat_counts, labels=repeat_labels, autopct='%1.1f%%',
            colors=['#3498db', '#e74c3c'], startangle=90, explode=(0, 0.05))
axes[0].set_title('New vs Repeat Customers')

purchase_dist = customer_features['purchase_count'].value_counts().sort_index().head(10)
axes[1].bar(purchase_dist.index.astype(str), purchase_dist.values, color='#3498db', edgecolor='white')
axes[1].set_xlabel('Number of Orders')
axes[1].set_ylabel('Number of Customers')
axes[1].set_title('Customer Purchase Frequency Distribution')

plt.tight_layout()
plt.show()

print(f"Total unique customers: {len(customer_features):,}")
print(f"Repeat customers:       {repeat_counts.get(1, 0):,} ({repeat_counts.get(1, 0)/len(customer_features)*100:.1f}%)")
print(f"One-time customers:     {repeat_counts.get(0, 0):,} ({repeat_counts.get(0, 0)/len(customer_features)*100:.1f}%)")

#### 5.1.2 High-Value vs Low-Value Customers

In [ ]:
# Segment customers by total spend using quartiles
customer_features['value_segment'] = pd.qcut(
    customer_features['total_spend'], q=4,
    labels=['Low Value', 'Mid-Low Value', 'Mid-High Value', 'High Value']
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seg_counts = customer_features['value_segment'].value_counts()
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
axes[0].bar(seg_counts.index.astype(str), seg_counts.values, color=colors, edgecolor='white')
axes[0].set_title('Customer Value Segments (by Total Spend)')
axes[0].set_ylabel('Number of Customers')

seg_revenue = customer_features.groupby('value_segment', observed=True)['total_spend'].sum()
axes[1].pie(seg_revenue, labels=seg_revenue.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Revenue Contribution by Customer Segment')

plt.tight_layout()
plt.show()

display(customer_features.groupby('value_segment', observed=True).agg(
    count=('customer_unique_id', 'count'),
    avg_spend=('total_spend', 'mean'),
    avg_orders=('purchase_count', 'mean'),
    avg_review=('avg_review_score', 'mean')
).round(2))

#### 5.1.3 Geographic Distribution of Customers

In [ ]:
state_customers = customer_features['customer_state'].value_counts().head(15)
state_revenue = customer_features.groupby('customer_state')['total_spend'].sum().sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(state_customers.index[::-1], state_customers.values[::-1], color='#3498db', edgecolor='white')
axes[0].set_xlabel('Number of Customers')
axes[0].set_title('Top 15 States by Customer Count')

axes[1].barh(state_revenue.index[::-1], state_revenue.values[::-1], color='#e74c3c', edgecolor='white')
axes[1].set_xlabel('Total Revenue (R$)')
axes[1].set_title('Top 15 States by Revenue')

plt.tight_layout()
plt.show()

# Top 10 cities
top_cities = customer_features['customer_city'].value_counts().head(10)
fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(top_cities.index[::-1], top_cities.values[::-1], color='#9b59b6', edgecolor='white')
ax.set_xlabel('Number of Customers')
ax.set_title('Top 10 Cities by Customer Count')
plt.tight_layout()
plt.show()

### 5.2 Revenue and Order Analysis

#### 5.2.1 Monthly Revenue and Order Volume Trends

In [ ]:
# Aggregate at the order level (deduplicate since master has item-level rows)
order_level = master.drop_duplicates(subset='order_id')

monthly = order_level.groupby('order_month').agg(
    revenue=('payment_value_total', 'sum'),
    order_count=('order_id', 'count'),
    avg_order_value=('payment_value_total', 'mean')
).reset_index()
monthly['order_month_dt'] = monthly['order_month'].dt.to_timestamp()

fig, ax1 = plt.subplots(figsize=(14, 6))

color_rev = '#2ecc71'
color_ord = '#3498db'

ax1.bar(monthly['order_month_dt'], monthly['revenue'], width=20,
        color=color_rev, alpha=0.7, label='Monthly Revenue (R$)')
ax1.set_xlabel('Month')
ax1.set_ylabel('Revenue (R$)', color=color_rev)
ax1.tick_params(axis='y', labelcolor=color_rev)
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.plot(monthly['order_month_dt'], monthly['order_count'],
         color=color_ord, linewidth=2, marker='o', markersize=5, label='Order Count')
ax2.set_ylabel('Number of Orders', color=color_ord)
ax2.tick_params(axis='y', labelcolor=color_ord)

fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.95))
plt.title('Monthly Revenue and Order Volume Trends')
plt.tight_layout()
plt.show()

#### 5.2.2 Peak Sales Periods — Day of Week & Hour of Day

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow = order_level['order_day_of_week'].value_counts().reindex(day_order)
axes[0].bar(dow.index, dow.values, color='#3498db', edgecolor='white')
axes[0].set_title('Orders by Day of Week')
axes[0].set_ylabel('Number of Orders')
axes[0].tick_params(axis='x', rotation=45)

hourly = order_level['order_hour'].value_counts().sort_index()
axes[1].plot(hourly.index, hourly.values, color='#e74c3c', linewidth=2, marker='o', markersize=4)
axes[1].fill_between(hourly.index, hourly.values, alpha=0.2, color='#e74c3c')
axes[1].set_title('Orders by Hour of Day')
axes[1].set_xlabel('Hour (0-23)')
axes[1].set_ylabel('Number of Orders')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.show()

#### 5.2.3 Payment Type Analysis

In [ ]:
pay_counts = payments['payment_type'].value_counts()
pay_revenue = payments.groupby('payment_type')['payment_value'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(pay_counts, labels=pay_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2'), startangle=90)
axes[0].set_title('Payment Type Distribution (by Count)')

axes[1].bar(pay_revenue.index, pay_revenue.values, color=sns.color_palette('Set2'), edgecolor='white')
axes[1].set_title('Revenue by Payment Type')
axes[1].set_ylabel('Total Revenue (R$)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### 5.3 Product Analysis

#### 5.3.1 Top-Selling Product Categories

In [ ]:
cat_sales = master.groupby('product_category_name_english').agg(
    order_count=('order_id', 'nunique'),
    items_sold=('order_item_id', 'count'),
    total_revenue=('price', 'sum'),
    avg_price=('price', 'mean'),
    avg_review=('review_score', 'mean')
).sort_values('items_sold', ascending=False)

top_15_cats = cat_sales.head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].barh(top_15_cats.index[::-1], top_15_cats['items_sold'].values[::-1],
             color='#3498db', edgecolor='white')
axes[0].set_xlabel('Items Sold')
axes[0].set_title('Top 15 Product Categories by Items Sold')

axes[1].barh(top_15_cats.index[::-1], top_15_cats['total_revenue'].values[::-1],
             color='#e74c3c', edgecolor='white')
axes[1].set_xlabel('Total Revenue (R$)')
axes[1].set_title('Top 15 Product Categories by Revenue')

plt.tight_layout()
plt.show()

#### 5.3.2 Product Price Distribution and Demand

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(master['price'], bins=50, color='#9b59b6', edgecolor='white', range=(0, 500))
axes[0].set_xlabel('Price (R$)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Product Price Distribution (capped at R$500)')

# Average price by top categories
top_cat_price = cat_sales.head(15)['avg_price'].sort_values()
axes[1].barh(top_cat_price.index, top_cat_price.values, color='#f39c12', edgecolor='white')
axes[1].set_xlabel('Average Price (R$)')
axes[1].set_title('Avg Price — Top 15 Categories')

plt.tight_layout()
plt.show()

### 5.4 Seller Analysis

In [ ]:
seller_perf = master.groupby('seller_id').agg(
    orders_fulfilled=('order_id', 'nunique'),
    items_sold=('order_item_id', 'count'),
    total_revenue=('price', 'sum'),
    avg_review=('review_score', 'mean'),
    seller_state=('seller_state', 'first')
).sort_values('total_revenue', ascending=False).reset_index()

print(f"Total active sellers: {len(seller_perf):,}")
print(f"\nTop 10 sellers by revenue:")
display(seller_perf.head(10)[['seller_id', 'orders_fulfilled', 'items_sold',
                               'total_revenue', 'avg_review', 'seller_state']])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Revenue distribution across sellers
axes[0].hist(seller_perf['total_revenue'], bins=50, color='#3498db', edgecolor='white')
axes[0].set_xlabel('Total Revenue (R$)')
axes[0].set_ylabel('Number of Sellers')
axes[0].set_title('Seller Revenue Distribution')

# Revenue concentration (top 20% of sellers)
seller_perf_sorted = seller_perf.sort_values('total_revenue', ascending=False)
cumulative_rev = seller_perf_sorted['total_revenue'].cumsum() / seller_perf_sorted['total_revenue'].sum() * 100
seller_pct = np.arange(1, len(cumulative_rev) + 1) / len(cumulative_rev) * 100
axes[1].plot(seller_pct, cumulative_rev, color='#e74c3c', linewidth=2)
axes[1].axhline(y=80, color='gray', linestyle='--', alpha=0.7)
axes[1].axvline(x=20, color='gray', linestyle='--', alpha=0.7)
axes[1].set_xlabel('% of Sellers')
axes[1].set_ylabel('% of Revenue (Cumulative)')
axes[1].set_title('Revenue Concentration (Pareto)')

# Seller distribution by state
seller_state = seller_perf['seller_state'].value_counts().head(10)
axes[2].barh(seller_state.index[::-1], seller_state.values[::-1], color='#2ecc71', edgecolor='white')
axes[2].set_xlabel('Number of Sellers')
axes[2].set_title('Top 10 States by Seller Count')

plt.tight_layout()
plt.show()

### 5.5 Review and Customer Satisfaction Analysis

#### 5.5.1 Review Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

score_dist = order_level['review_score'].value_counts().sort_index()
colors_scores = ['#e74c3c', '#f39c12', '#f1c40f', '#2ecc71', '#27ae60']
axes[0].bar(score_dist.index, score_dist.values, color=colors_scores, edgecolor='white')
axes[0].set_xlabel('Review Score')
axes[0].set_ylabel('Number of Orders')
axes[0].set_title('Distribution of Review Scores')
for i, (score, count) in enumerate(zip(score_dist.index, score_dist.values)):
    axes[0].text(score, count + 200, f'{count/len(order_level)*100:.1f}%', ha='center', fontsize=10)

# Monthly average review score trend
monthly_review = order_level.groupby('order_month')['review_score'].mean().reset_index()
monthly_review['order_month_dt'] = monthly_review['order_month'].dt.to_timestamp()
axes[1].plot(monthly_review['order_month_dt'], monthly_review['review_score'],
             color='#e74c3c', linewidth=2, marker='o', markersize=4)
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Average Review Score')
axes[1].set_title('Monthly Average Review Score Trend')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylim(3, 5)

plt.tight_layout()
plt.show()

#### 5.5.2 Relationship Between Delivery Time and Ratings

In [ ]:
delivered = order_level[order_level['order_status'] == 'delivered'].dropna(subset=['delivery_time_days', 'review_score'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot: delivery time by review score
delivered.boxplot(column='delivery_time_days', by='review_score', ax=axes[0],
                  patch_artist=True, boxprops=dict(facecolor='#3498db', alpha=0.7))
axes[0].set_xlabel('Review Score')
axes[0].set_ylabel('Delivery Time (days)')
axes[0].set_title('Delivery Time by Review Score')
axes[0].set_ylim(0, 60)
plt.sca(axes[0])
plt.title('Delivery Time by Review Score')

# Late vs On-time delivery impact on ratings
delivered['delivery_status'] = np.where(delivered['delivery_delay_days'] >= 0, 'On Time / Early', 'Late')
delivery_review = delivered.groupby('delivery_status')['review_score'].mean()
axes[1].bar(delivery_review.index, delivery_review.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[1].set_ylabel('Average Review Score')
axes[1].set_title('Average Review Score: On-Time vs Late Delivery')
for i, (status, score) in enumerate(zip(delivery_review.index, delivery_review.values)):
    axes[1].text(i, score + 0.05, f'{score:.2f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.suptitle('')
plt.show()

print(f"Correlation between delivery time and review score: "
      f"{delivered['delivery_time_days'].corr(delivered['review_score']):.3f}")

#### 5.5.3 Dissatisfaction Patterns (Low-rated Orders)

In [ ]:
low_rated = delivered[delivered['review_score'] <= 2]
high_rated = delivered[delivered['review_score'] >= 4]

print("Dissatisfaction Analysis (Score 1-2 vs Score 4-5):\n")
print(f"{'Metric':<35s} {'Low (1-2)':>12s} {'High (4-5)':>12s}")
print("-" * 60)
print(f"{'Number of orders':<35s} {len(low_rated):>12,} {len(high_rated):>12,}")
print(f"{'Avg delivery time (days)':<35s} {low_rated['delivery_time_days'].mean():>12.1f} {high_rated['delivery_time_days'].mean():>12.1f}")
print(f"{'% Late deliveries':<35s} {(low_rated['delivery_delay_days'] < 0).mean()*100:>11.1f}% {(high_rated['delivery_delay_days'] < 0).mean()*100:>11.1f}%")
print(f"{'Avg order value (R$)':<35s} {low_rated['payment_value_total'].mean():>12.2f} {high_rated['payment_value_total'].mean():>12.2f}")

# Category-wise dissatisfaction
cat_low = low_rated['product_category_name_english'].value_counts().head(10)
cat_high = high_rated['product_category_name_english'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(cat_low.index[::-1], cat_low.values[::-1], color='#e74c3c', edgecolor='white')
axes[0].set_title('Top 10 Categories with Low Ratings (1-2)')
axes[0].set_xlabel('Number of Orders')

axes[1].barh(cat_high.index[::-1], cat_high.values[::-1], color='#2ecc71', edgecolor='white')
axes[1].set_title('Top 10 Categories with High Ratings (4-5)')
axes[1].set_xlabel('Number of Orders')

plt.tight_layout()
plt.show()

### 5.6 Advanced Visualizations

#### 5.6.1 Correlation Heatmap

In [ ]:
corr_cols = ['price', 'freight_value', 'payment_value_total', 'review_score',
             'delivery_time_days', 'delivery_delay_days', 'items_per_order',
             'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

corr_data = order_level[corr_cols].dropna()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap of Key Numerical Features')
plt.tight_layout()
plt.show()

#### 5.6.2 Outlier Detection — Box Plots

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

bp_data = [
    ('price', 'Product Price (R$)', '#3498db'),
    ('freight_value', 'Freight Value (R$)', '#e74c3c'),
    ('delivery_time_days', 'Delivery Time (days)', '#2ecc71'),
    ('payment_value_total', 'Payment Value (R$)', '#f39c12'),
]

for ax, (col, title, color) in zip(axes, bp_data):
    data = order_level[col].dropna()
    bp = ax.boxplot(data, patch_artist=True, boxprops=dict(facecolor=color, alpha=0.7))
    ax.set_title(title)
    ax.set_ylabel(col)
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    outlier_count = ((data < q1 - 1.5 * iqr) | (data > q3 + 1.5 * iqr)).sum()
    ax.text(1, data.max() * 0.95, f'Outliers: {outlier_count:,}', ha='center', fontsize=9)

plt.suptitle('Outlier Detection Using Box Plots', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

#### 5.6.3 Freight Value vs Product Weight

In [ ]:
sample = master.dropna(subset=['product_weight_g', 'freight_value']).sample(5000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(sample['product_weight_g'], sample['freight_value'],
                     c=sample['review_score'], cmap='RdYlGn', alpha=0.5, s=15, edgecolors='none')
plt.colorbar(scatter, label='Review Score')
ax.set_xlabel('Product Weight (g)')
ax.set_ylabel('Freight Value (R$)')
ax.set_title('Freight Value vs Product Weight (colored by Review Score)')
ax.set_xlim(0, 15000)
ax.set_ylim(0, 150)
plt.tight_layout()
plt.show()

#### 5.6.4 Review Score Distribution by Top Product Categories

In [ ]:
top_10_categories = master['product_category_name_english'].value_counts().head(10).index
cat_review_data = master[master['product_category_name_english'].isin(top_10_categories)]

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=cat_review_data, x='product_category_name_english', y='review_score',
            palette='Set2', ax=ax, order=top_10_categories)
ax.set_xlabel('Product Category')
ax.set_ylabel('Review Score')
ax.set_title('Review Score Distribution — Top 10 Product Categories')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

---
## Step 7: Business Insights and Recommendations

### 7.1 Key Findings Summary

In [ ]:
total_revenue = order_level['payment_value_total'].sum()
total_orders = order_level['order_id'].nunique()
total_customers = customer_features['customer_unique_id'].nunique()
repeat_rate = customer_features['is_repeat_customer'].mean() * 100
avg_delivery = delivered['delivery_time_days'].mean()
avg_review = order_level['review_score'].mean()
late_pct = (delivered['delivery_delay_days'] < 0).mean() * 100

print("=" * 70)
print("           E-COMMERCE BUSINESS INTELLIGENCE SUMMARY")
print("=" * 70)
print(f"""
  Total Revenue:              R$ {total_revenue:>14,.2f}
  Total Orders:               {total_orders:>14,}
  Total Unique Customers:     {total_customers:>14,}
  Repeat Customer Rate:       {repeat_rate:>13.1f}%
  Avg Delivery Time:          {avg_delivery:>12.1f} days
  Late Delivery Rate:         {late_pct:>13.1f}%
  Avg Review Score:           {avg_review:>14.2f} / 5
""")
print("=" * 70)

### 7.2 Top Revenue-Driving Factors

In [ ]:
print("TOP REVENUE DRIVERS:\n")

# By state
top_states = customer_features.groupby('customer_state')['total_spend'].sum().sort_values(ascending=False).head(5)
print("  Top 5 states by revenue:")
for state, rev in top_states.items():
    print(f"    {state}: R$ {rev:,.2f} ({rev/total_revenue*100:.1f}%)")

# By category
print("\n  Top 5 product categories by revenue:")
top_cats = cat_sales.head(5)
for cat, row in top_cats.iterrows():
    print(f"    {cat}: R$ {row['total_revenue']:,.2f}")

# By payment type
print(f"\n  Credit card dominates payments: {(payments['payment_type']=='credit_card').mean()*100:.1f}% of all transactions")

# Seller concentration
top_20_pct = int(len(seller_perf) * 0.2)
top_20_rev = seller_perf.head(top_20_pct)['total_revenue'].sum()
print(f"\n  Top 20% of sellers generate {top_20_rev/seller_perf['total_revenue'].sum()*100:.1f}% of revenue (Pareto effect)")

### 7.3 Customer Behavior Patterns

In [ ]:
print("CUSTOMER BEHAVIOR PATTERNS:\n")

print(f"  1. RETENTION CHALLENGE: Only {repeat_rate:.1f}% of customers are repeat buyers.")
print(f"     The vast majority make a single purchase, indicating a need for")
print(f"     loyalty programs and re-engagement campaigns.\n")

avg_spend_repeat = customer_features[customer_features['is_repeat_customer']==1]['total_spend'].mean()
avg_spend_new = customer_features[customer_features['is_repeat_customer']==0]['total_spend'].mean()
print(f"  2. REPEAT CUSTOMER VALUE: Repeat customers spend R${avg_spend_repeat:.2f} on avg")
print(f"     vs R${avg_spend_new:.2f} for one-time customers ({avg_spend_repeat/avg_spend_new:.1f}x more).\n")

peak_hour = order_level['order_hour'].value_counts().idxmax()
peak_day = order_level['order_day_of_week'].value_counts().idxmax()
print(f"  3. PEAK SHOPPING: Most orders are placed on {peak_day}s around {peak_hour}:00.")
print(f"     This informs optimal timing for marketing campaigns.\n")

high_val_rev = customer_features[customer_features['value_segment']=='High Value']['total_spend'].sum()
print(f"  4. HIGH-VALUE SEGMENT: Top 25% of customers contribute")
print(f"     R${high_val_rev:,.2f} ({high_val_rev/customer_features['total_spend'].sum()*100:.1f}% of total revenue).")

### 7.4 Operational Inefficiencies

In [ ]:
print("OPERATIONAL INEFFICIENCIES:\n")

print(f"  1. DELIVERY DELAYS: {late_pct:.1f}% of delivered orders arrived late.")
print(f"     Late orders have an avg review score of {delivered[delivered['delivery_delay_days']<0]['review_score'].mean():.2f}")
print(f"     vs {delivered[delivered['delivery_delay_days']>=0]['review_score'].mean():.2f} for on-time orders.\n")

long_delivery = (delivered['delivery_time_days'] > 30).mean() * 100
print(f"  2. LONG DELIVERIES: {long_delivery:.1f}% of orders take more than 30 days to deliver.\n")

cancelled_pct = (orders['order_status'] == 'canceled').mean() * 100
unavail_pct = (orders['order_status'] == 'unavailable').mean() * 100
print(f"  3. ORDER FAILURES: {cancelled_pct:.2f}% cancelled, {unavail_pct:.2f}% unavailable.\n")

# Freight cost as % of order value
freight_ratio = (master['freight_value'] / master['price']).median() * 100
print(f"  4. FREIGHT BURDEN: Median freight cost is {freight_ratio:.1f}% of product price,")
print(f"     which may deter price-sensitive customers.")

### 7.5 Strategic Recommendations

Based on the analysis above, the following strategic recommendations are proposed:

**1. Improve Customer Retention**
- The repeat customer rate is extremely low. Implement a loyalty rewards program, personalized email campaigns, and post-purchase engagement to convert one-time buyers into repeat customers.
- Since repeat customers spend significantly more, even a small improvement in retention would yield substantial revenue gains.

**2. Optimize Delivery Operations**
- Delivery time is the single strongest predictor of low review scores. Prioritize logistics improvements, especially for regions far from seller hubs.
- Establish regional fulfillment centers or partner with local carriers in states with high order volumes but long delivery times.
- Set realistic delivery estimates to avoid customer disappointment; late deliveries hurt satisfaction far more than slightly longer but accurate estimates.

**3. Leverage Geographic Concentration**
- Sao Paulo (SP) dominates both customer and seller presence. Expand marketing and seller onboarding in underserved states to diversify the customer base.
- Consider state-specific promotions for high-potential but currently low-penetration markets.

**4. Reduce Freight Cost Impact**
- High freight-to-price ratios deter budget-conscious shoppers. Explore free-shipping thresholds, bundled shipping discounts, or subsidized freight for high-margin categories.

**5. Invest in High-Performing Categories**
- Focus marketing spend on the top revenue-generating categories while improving review scores for categories with high sales volume but lower satisfaction.
- Curate quality standards for sellers in categories with the most dissatisfaction complaints.

**6. Seller Quality Program**
- Revenue is concentrated among a small subset of sellers (Pareto effect). Develop a seller training and quality certification program to elevate mid-tier sellers.
- Underperforming sellers should be flagged for review to maintain platform quality.

**7. Time-Based Marketing**
- Schedule promotional campaigns, flash sales, and email blasts during peak activity windows (identified as weekday afternoons/evenings) to maximize conversion rates.

---
## Conclusion

This project successfully integrated 9 separate data sources into a unified Customer 360 analytical framework. Through systematic data cleaning, merging, feature engineering, and exploratory analysis, we uncovered key patterns in customer behavior, revenue drivers, product performance, seller dynamics, and delivery operations. The insights generated are data-driven and supported by visual evidence, providing a foundation for strategic business decisions in the e-commerce domain.